# 01 - Data Generator
Generates simulated UPI transaction JSON files with intentional data-quality issues
and fraud-pattern anomalies, and writes them into the Unity Catalog Volume.

In [0]:
import json
import random
import string
import uuid
from datetime import datetime, timedelta

VOLUME_PATH = "/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming"
dbutils.fs.mkdirs(VOLUME_PATH)

NUM_FILES = 30
RECORDS_PER_FILE = 500

BANKS = ["HDFC", "ICICI", "SBI", "AXIS", "KOTAK", "PNB", "YES_BANK"]
STATUSES_VALID = ["SUCCESS", "FAILED", "TIMEOUT", "PENDING"]
STATUSES_MESSY = ["success", "Success", " SUCCESS", "failed ", "TimeOut", "SUCCESS", "FAILED"]
SUSPICIOUS_IP_PREFIXES = ["185.220.", "45.153.", "89.248."]  # simulated "known-bad" ranges

def random_upi_id():
    return f"{''.join(random.choices(string.ascii_lowercase, k=6))}{random.randint(1,999)}@{random.choice(['okhdfc','oksbi','okicici','okaxis'])}"

def random_ip(suspicious=False):
    if suspicious:
        return random.choice(SUSPICIOUS_IP_PREFIXES) + f"{random.randint(1,254)}.{random.randint(1,254)}"
    return f"{random.randint(1,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

def random_phone():
    return "9" + "".join(random.choices(string.digits, k=9))

def base_record(ts):
    return {
        "transaction_id": str(uuid.uuid4()),
        "timestamp": ts.isoformat(),
        "sender_upi_id": random_upi_id(),
        "receiver_upi_id": random_upi_id(),
        "customer_phone": random_phone(),
        "bank_name": random.choice(BANKS),
        "amount": round(random.uniform(10, 50000), 2),
        "status": random.choice(STATUSES_VALID),
        "ip_address": random_ip(),
    }

def corrupt_record(rec):
    """Randomly injects nulls, whitespace, type mismatches, negative amounts."""
    choice = random.choice(["null_id", "whitespace", "bad_type", "negative_amount", "messy_status", "null_ip"])
    if choice == "null_id":
        rec["transaction_id"] = None
    elif choice == "whitespace":
        rec["sender_upi_id"] = f"   {rec['sender_upi_id']}   "
        rec["receiver_upi_id"] = f"{rec['receiver_upi_id']}\t"
    elif choice == "bad_type":
        rec["amount"] = str(rec["amount"]) + " INR"  # amount as messy string
    elif choice == "negative_amount":
        rec["amount"] = -abs(rec["amount"])
    elif choice == "messy_status":
        rec["status"] = random.choice(STATUSES_MESSY)
    elif choice == "null_ip":
        rec["ip_address"] = None
    return rec

def fraud_burst(start_ts, sender_upi_id, ip):
    """Generates a velocity-spike pattern: many transactions from the same sender/IP in a short window."""
    burst = []
    for i in range(random.randint(6, 10)):
        ts = start_ts + timedelta(seconds=random.randint(1, 400))
        rec = base_record(ts)
        rec["sender_upi_id"] = sender_upi_id
        rec["ip_address"] = ip
        rec["amount"] = round(random.uniform(5000, 20000), 2)
        burst.append(rec)
    return burst

def high_value_fraud(ts):
    rec = base_record(ts)
    rec["amount"] = round(random.uniform(200001, 500000), 2)  # unusually high
    rec["ip_address"] = random_ip(suspicious=True)
    return rec

In [0]:
random.seed(42)
total_records = 0
start_time = datetime(2026, 8, 1, 0, 0, 0)

for file_idx in range(NUM_FILES):
    records = []
    file_start = start_time + timedelta(hours=file_idx)

    for _ in range(RECORDS_PER_FILE):
        ts = file_start + timedelta(seconds=random.randint(0, 3599))
        rec = base_record(ts)

        roll = random.random()
        if roll < 0.10:                       # ~10% corrupted / bad-quality records
            rec = corrupt_record(rec)
        elif roll < 0.13:                      # ~3% high-value fraud pattern
            rec = high_value_fraud(ts)
        elif roll < 0.15:                      # ~2% bank timeout scenario
            rec["status"] = "TIMEOUT"

        records.append(rec)

    # inject a velocity-spike fraud burst into ~1 in 3 files
    if file_idx % 3 == 0:
        fraud_sender = random_upi_id()
        fraud_ip = random_ip(suspicious=True)
        records.extend(fraud_burst(file_start, fraud_sender, fraud_ip))

    file_path = f"{VOLUME_PATH}/upi_transactions_{file_idx:03d}.json"
    content = "\n".join(json.dumps(r) for r in records)
    dbutils.fs.put(file_path, content, overwrite=True)
    total_records += len(records)

print(f"Generated {NUM_FILES} files, {total_records} total records into {VOLUME_PATH}/")

Wrote 149646 bytes.
Wrote 147798 bytes.
Wrote 147478 bytes.
Wrote 149797 bytes.
Wrote 147762 bytes.
Wrote 147931 bytes.
Wrote 149612 bytes.
Wrote 147705 bytes.
Wrote 147829 bytes.
Wrote 149512 bytes.
Wrote 147749 bytes.
Wrote 147757 bytes.
Wrote 150314 bytes.
Wrote 147751 bytes.
Wrote 147896 bytes.
Wrote 150353 bytes.
Wrote 147856 bytes.
Wrote 147697 bytes.
Wrote 149651 bytes.
Wrote 147860 bytes.
Wrote 147836 bytes.
Wrote 150161 bytes.
Wrote 147608 bytes.
Wrote 147861 bytes.
Wrote 149928 bytes.
Wrote 147784 bytes.
Wrote 147828 bytes.
Wrote 149586 bytes.
Wrote 147685 bytes.
Wrote 147841 bytes.
Generated 30 files, 15070 total records into /Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/


### Verify

In [0]:
files = dbutils.fs.ls(VOLUME_PATH)
print(f"File count: {len(files)}")
display(files)

File count: 30


path,name,size,modificationTime
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_000.json,upi_transactions_000.json,149646,1786207947000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_001.json,upi_transactions_001.json,147798,1786207948000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_002.json,upi_transactions_002.json,147478,1786207948000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_003.json,upi_transactions_003.json,149797,1786207949000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_004.json,upi_transactions_004.json,147762,1786207949000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_005.json,upi_transactions_005.json,147931,1786207950000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_006.json,upi_transactions_006.json,149612,1786207950000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_007.json,upi_transactions_007.json,147705,1786207950000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_008.json,upi_transactions_008.json,147829,1786207951000
dbfs:/Volumes/sentinel_catalog/sentinel_schema/raw_volume/incoming/upi_transactions_009.json,upi_transactions_009.json,149512,1786207951000
